# Ch.00 — Data Acquisition and Collection *(Solution)*

> **Dataset:** SpaceX Falcon 9 launches — REST API v5 + Wikipedia HTML tables
> **Chapter doc:** [data-acquisition.md](data-acquisition.md)

Fully implemented. Run from top to bottom.

| Section | Functions |
|---------|-----------|
| § 1 REST API | `build_session`, `fetch_all_launches`, `LaunchRecord`, `fetch_validated_launches` |
| § 2 Scraping | `fetch_with_change_detection`, `scrape_wikipedia_launches` |
| § 3 Wrangling | `handle_landing_outcome`, `coerce_boolean`, `merge_with_priority`, `validate_launches` |
| § 4 Pipeline | `run_pipeline` |


In [ ]:
# %pip install requests pydantic pandas beautifulsoup4 lxml --quiet

import hashlib, json, logging, re
from datetime import datetime
from pathlib import Path
from typing import Optional

import pandas as pd
import requests
from bs4 import BeautifulSoup
from pydantic import BaseModel, field_validator
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

RAW_DIR       = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

LAUNCHES_URL = "https://api.spacexdata.com/v5/launches"
RAW_JSONL    = RAW_DIR / "launches.jsonl"
HASH_STORE   = RAW_DIR / ".content_hashes.json"

print("Setup complete.", RAW_DIR, PROCESSED_DIR)


---
## § 1 — REST API Data Collection

### HTTP status codes

| Code | Meaning | Action |
|------|---------|--------|
| 200 | OK | Parse and persist |
| 400 | Bad request | Fix request; **do not retry** |
| 429 | Rate limited | Back off; respect `Retry-After` |
| 500/503 | Server error | Retry with backoff |

### Exponential backoff

$$t_{wait} = \min\!\left(c \cdot 2^n + \varepsilon,\; t_{max}\right)$$

$n$ = attempt (0-indexed), $c=1$ s, $\varepsilon \sim \text{Uniform}(0,c)$ jitter prevents thundering herd.
`backoff_factor=1.0` → waits 0 s, 2 s, 4 s, 8 s, 16 s.

Reference: data-acquisition.md § 1 — "Rate limiting and exponential backoff"

In [ ]:
def build_session(retries=5, backoff_factor=1.0,
                  status_forcelist=(429, 500, 502, 503, 504)):
    """HTTP session with automatic retry and exponential backoff."""
    session = requests.Session()
    retry = Retry(
        total=retries,
        backoff_factor=backoff_factor,
        status_forcelist=status_forcelist,
        respect_retry_after_header=True,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://",  adapter)
    return session

session = build_session()
print("Session:", session)


### Pagination — the silent truncation problem

The SpaceX v5 API defaults to **10** results per call and caps at **100**.
Calling `/v5/launches` with no params returns only the 10 most recent — not the 200+ corpus.

**Offset pattern:** `?offset=0&limit=100` → `?offset=100&limit=100` → ... → empty list = done.

 data-acquisition.md § 1 — "Pagination: the silent truncation problem"

In [ ]:
def fetch_all_launches(url=LAUNCHES_URL, output=RAW_JSONL, limit=100):
    """Full-refresh fetch to JSONL using offset pagination."""
    sess = build_session()
    output.parent.mkdir(parents=True, exist_ok=True)
    written, offset = 0, 0
    with open(output, "w", encoding="utf-8") as f:
        while True:
            resp = sess.get(url, params={"offset": offset, "limit": limit}, timeout=15)
            resp.raise_for_status()
            records = resp.json()
            if not records:
                break
            for r in records:
                f.write(json.dumps(r) + "\n")
            written += len(records)
            offset  += limit
            log.info(f"  fetched {written} records (offset={offset})")
    log.info(f"Total written: {written}")
    return written

# Preview one record's keys (no full fetch required)
sample = requests.get(LAUNCHES_URL, params={"limit": 1}, timeout=10).json()[0]
print("Keys:", list(sample.keys()))


### Schema validation — catching drift at ingest time

SpaceX v3 → v5: `launch_success` renamed to `success`.
A v3-era pipeline silently reads `None` for the target — no exception, just a useless model.

**Fix:** Pydantic validates every record at ingest. `ValidationError` fires immediately.

Key patterns:
- `Optional[bool] = None` — null is legitimate for upcoming launches
- `@field_validator("date_utc", mode="before")` — transform raw value before validation
- `model_dump(mode="json")` — JSON-safe output for JSONL

 data-acquisition.md § 1 — "Schema validation: catching drift at ingest time"

In [ ]:
class LaunchRecord(BaseModel):
    """Pydantic model for SpaceX v5 launch records.
    ValidationError fires at ingest if a required field is absent — making
    schema drift a loud error rather than a silent null in the target column.
    """
    id:       str
    name:     str
    date_utc: datetime
    success:  Optional[bool] = None   # null for upcoming launches — legitimate
    upcoming: bool
    cores:    list

    @field_validator("date_utc", mode="before")
    @classmethod
    def parse_date(cls, v):
        if isinstance(v, str):
            return datetime.fromisoformat(v.rstrip("Z"))
        return v


def fetch_validated_launches(url=LAUNCHES_URL, output=RAW_JSONL, limit=100):
    """Paginated fetch with per-record Pydantic validation."""
    from pydantic import ValidationError
    sess = build_session()
    output.parent.mkdir(parents=True, exist_ok=True)
    written, skipped, offset = 0, 0, 0
    with open(output, "w", encoding="utf-8") as f:
        while True:
            resp = sess.get(url, params={"offset": offset, "limit": limit}, timeout=15)
            resp.raise_for_status()
            records = resp.json()
            if not records:
                break
            for raw in records:
                try:
                    launch = LaunchRecord(**raw)
                    f.write(json.dumps(launch.model_dump(mode="json")) + "\n")
                    written += 1
                except ValidationError as e:
                    log.warning(f"Skipping {raw.get('id')}: {e}")
                    skipped += 1
            offset += limit
    log.info(f"Written: {written} | Skipped: {skipped}")
    return written


# Test against a live record
raw = requests.get(LAUNCHES_URL, params={"limit": 1}, timeout=10).json()[0]
v   = LaunchRecord(**raw)
print(f"id={v.id}  name={v.name}  success={v.success}  upcoming={v.upcoming}")


In [ ]:
# Fetch all launches into a DataFrame for exploration
records, offset = [], 0
while True:
    batch = requests.get(LAUNCHES_URL,
                         params={"offset": offset, "limit": 100}, timeout=15).json()
    if not batch:
        break
    records.extend(batch)
    offset += 100

df_api = pd.DataFrame([{
    "flight_number": r.get("flight_number"),
    "name":          r.get("name"),
    "date_utc":      r.get("date_utc"),
    "success":       r.get("success"),
    "upcoming":      r.get("upcoming"),
} for r in records])
df_api["date_utc"] = pd.to_datetime(df_api["date_utc"], utc=True)

print(f"Total launches : {len(df_api)}")
print(f"Success rate   : {df_api['success'].mean():.1%}  (excludes null)")
print(f"Null success   : {df_api['success'].isna().sum()}  (upcoming / unknown)")
df_api.head()


---
## § 2 — Web Scraping

SpaceX launches 2010–2014 predate the API. Only Wikipedia has early Falcon 9 test outcomes.

### Parsing hierarchy

```
URL → requests.get() → BeautifulSoup → .find("table", {"class":"wikitable"}) → pd.read_html()
```

**CSS selector fragility:** `soup.find("table", {"class":"wikitable"})` is stable —
`wikitable` is an established MediaWiki convention. Never use positional indexing
(`soup.find_all("table")[3]`) — it breaks on the first table addition.

### Change detection

Hash raw HTML bytes. On first run: store hash. On subsequent runs:
if hash changed → **raise ValueError before parsing** — loud failure, not silent corruption.

 data-acquisition.md § 2 — "Change detection: hash, don't assume"

In [ ]:
WIKI_URL = "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches"
def fetch_with_change_detection(url, name):
    """Fetch URL; raise ValueError if content hash differs from last known state."""
    resp = requests.get(url, timeout=20, headers={"User-Agent": "research-bot/1.0"})
    resp.raise_for_status()
    current_hash = hashlib.sha256(resp.content).hexdigest()
    hashes    = json.loads(HASH_STORE.read_text()) if HASH_STORE.exists() else {}
    last_hash = hashes.get(name)
    if last_hash and last_hash != current_hash:
        raise ValueError(
            f"Content hash mismatch for '{name}':\n"
            f"  stored : {last_hash[:12]}...\n"
            f"  current: {current_hash[:12]}...\n"
            "Page may have changed. Inspect before proceeding."
        )
    hashes[name] = current_hash
    HASH_STORE.write_text(json.dumps(hashes, indent=2))
    return resp.content

print("fetch_with_change_detection() defined.")


In [ ]:
WIKI_RAW = RAW_DIR / "html" / "wiki_launches.html"
WIKI_RAW.parent.mkdir(parents=True, exist_ok=True)


def scrape_wikipedia_launches(url=WIKI_URL):
    """Fetch Wikipedia Falcon 9 launch table; return cleaned DataFrame."""
    html_bytes = fetch_with_change_detection(url, "wiki_falcon9_launches")
    WIKI_RAW.write_bytes(html_bytes)

    soup  = BeautifulSoup(html_bytes, "html.parser")
    table = soup.find("table", {"class": "wikitable"})
    if table is None:
        raise RuntimeError("No wikitable found — page structure may have changed.")

    df = pd.read_html(str(table))[0]

    # Flatten multi-level column headers (common in Wikipedia tables)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            " ".join(filter(None, map(str, col))).strip()
            for col in df.columns
        ]

    df = df.replace(r"\[.*?\]", "", regex=True)   # strip [1], [note 1]
    df = df.replace(r"^\s*$", pd.NA, regex=True)    # whitespace-only → NA
    df = df[df.apply(lambda r: r.nunique(dropna=False) > 1, axis=1)]  # drop footnote rows
    df = df.dropna(how="all").reset_index(drop=True)
    return df


# Uncomment to run (hits Wikipedia):
# df_wiki = scrape_wikipedia_launches()
# print(df_wiki.shape, df_wiki.columns.tolist())
# df_wiki.head(3)
print("scrape_wikipedia_launches() defined. Uncomment last 3 lines to fetch.")


---
## § 3 — Data Wrangling

### Missing value taxonomy: MCAR / MAR / MNAR

| Mechanism | Definition | SpaceX example | Treatment |
|-----------|-----------|----------------|----------|
| **MCAR** | Missing independent of all variables | 2% API dropout from timeouts | Drop rows (< 5%) or impute with median |
| **MAR** | Missing depends on *other observed* columns | `payload_mass_kg` missing more for early launches (correlates with `year`) | Group imputation |
| **MNAR** | Missing depends on *the value itself* | `landing_outcome` absent when landing **not attempted** | Indicator columns — **do not impute** |

**MNAR critical insight:** imputing `landing_success = False` for not-attempted launches
teaches the model those configurations *failed*. They simply weren't attempted.
Three correct states: attempted+succeeded, attempted+failed, not attempted.

 data-acquisition.md § 3 — "Missing value taxonomy: MCAR, MAR, MNAR"

In [ ]:
def handle_landing_outcome(df):
    """MNAR treatment: split 'landing_outcome' into two indicator columns."""
    df = df.copy()
    _SUCCESS = {"success", "true", "1", "landed", "successful"}
    _FAILURE = {"failure", "false", "0", "failed", "crash", "explosion"}

    df["landing_attempted"] = df["landing_outcome"].notna()

    def parse_outcome(v):
        if pd.isna(v):
            return pd.NA
        s = str(v).lower().strip()
        if s in _SUCCESS:
            return True
        if s in _FAILURE:
            return False
        return pd.NA   # unrecognised — not assumed failure

    df["landing_success"] = (
        df["landing_outcome"].map(parse_outcome).astype(pd.BooleanDtype())
    )
    return df


# Demo with synthetic data
demo = pd.DataFrame({
    "flight":          [1,        2,        3,    4,        5],
    "landing_outcome": ["Success","Failure", None,"success","Crash"],
})
print(handle_landing_outcome(demo)[
    ["flight", "landing_outcome", "landing_attempted", "landing_success"]
])


In [ ]:
def coerce_boolean(series):
    """Normalise any boolean-ish string to pd.BooleanDtype.
    Unrecognised values → pd.NA (not assumed True or False).
    """
    _TRUE  = {"true", "yes", "1", "success", "successful", "landed"}
    _FALSE = {"false", "no", "0", "failure", "failed", "crash",
              "n/a", "none", "nan", ""}
    return (
        series.astype(str).str.lower().str.strip()
        .map(lambda v: True if v in _TRUE else (False if v in _FALSE else pd.NA))
        .astype(pd.BooleanDtype())
    )


def coerce_mass_kg(series):
    """Parse '9525 kg', '9,525', 9525.0 to float. errors='coerce' makes failures visible."""
    return pd.to_numeric(
        series.astype(str).str.replace(r"[^\d.]", "", regex=True),
        errors="coerce",
    )


# Demo
messy = pd.Series(["TRUE", "False", "1", "0", "Success", "N/A", None, "yes", "unknown"])
clean = coerce_boolean(messy)
print(pd.DataFrame({"raw": messy, "coerced": clean}).to_string(index=False))
print(f"\nNull (unrecognised): {clean.isna().sum()}")


### Deduplication and source-priority merging

The same launch can appear in API and Wikipedia with **different values** — one source was
retroactively corrected. Define priority per field, document it, use `combine_first()`.

`combine_first(other)` — takes left value when non-null, falls back to right.

### Validation contract

Write assertions **before** writing wrangling code.
If assertions fail, pipeline halts with a clear error — not a silent corrupt output.

 data-acquisition.md § 3 — "Deduplication" + "Validation before downstream use"

In [ ]:
def merge_with_priority(api_df, csv_df, key="flight_number"):
    """Outer join with explicit source priority per field.
    API wins for outcome fields; CSV wins for descriptive text.
    """
    merged = api_df.merge(csv_df, on=key, suffixes=("_api", "_csv"), how="outer")
    if "success_api" in merged.columns:
        merged["success"] = merged["success_api"].combine_first(merged["success_csv"])
        merged.drop(columns=["success_api", "success_csv"], errors="ignore", inplace=True)
    if "launch_site_csv" in merged.columns:
        merged["launch_site"] = merged["launch_site_csv"].combine_first(
            merged.get("launch_site_api")
        )
        merged.drop(columns=["launch_site_api", "launch_site_csv"],
                    errors="ignore", inplace=True)
    return merged


def validate_launches(df):
    """Assert data quality invariants before writing to data/processed/."""
    assert len(df) >= 50, f"Too few records: {len(df)} (expected >= 50)"
    for col in ["flight_number", "name"]:
        if col in df.columns:
            n = df[col].isna().sum()
            assert n == 0, f"Required column '{col}' has {n} nulls"
    if "flight_number" in df.columns:
        dupes = df["flight_number"].dropna().duplicated().sum()
        assert dupes == 0, f"{dupes} duplicate flight_numbers"


try:
    validate_launches(df_api)
    print("Validation passed ✅")
except AssertionError as e:
    print(f"Validation failed: {e}")


---
## § 4 — Reproducible Pipeline

### Three enemies of reproducibility
1. **Mutable sources** — APIs return different data on different days
2. **Local state** — files the pipeline reads that aren't in version control
3. **Implicit ordering** — step 3 silently reads stale intermediate data

### Layout rule
```
data/
├── raw/ ← append-only; never overwritten
└── processed/ ← derivable from raw; disposable
```

**Idempotency:** running twice = identical output. Hash-gated fetches make this cheap.

 data-acquisition.md § 4 — "Idempotency" + "Raw vs processed"

In [ ]:
def run_pipeline():
    """Idempotent pipeline: fetch → validate schema → wrangle → validate → persist."""
    log.info("=== Pipeline start ===")

    # Stage 1 — fetch raw data
    fetched = fetch_validated_launches(output=RAW_JSONL)
    log.info(f"Stage 1: {fetched} records → {RAW_JSONL}")

    # Stage 2 — load raw
    df = pd.read_json(RAW_JSONL, lines=True)
    log.info(f"Stage 2 — raw shape: {df.shape} | nulls: {df.isna().sum().to_dict()}")

    # Stage 3 — wrangle
    df["date_utc"] = pd.to_datetime(df["date_utc"], utc=True)
    df["success"]  = coerce_boolean(df["success"].astype(str))
    df = df.drop_duplicates(subset=["flight_number"])
    log.info(f"Stage 3 — {len(df)} rows, {df.isna().sum().sum()} nulls remaining")

    # Stage 4 — validate
    try:
        validate_launches(df)
        log.info("Stage 4 — validation passed ✅")
    except AssertionError as e:
        log.error(f"Validation FAILED: {e}")
        raise

    # Stage 5 — persist
    out = PROCESSED_DIR / "launches_clean.parquet"
    df.to_parquet(out, index=False)
    log.info(f"=== Pipeline complete — {len(df)} rows → {out} ===")
    return df


# Uncomment to run the full pipeline:
# df_final = run_pipeline()
# df_final.head()
print("run_pipeline() defined. Uncomment last 2 lines to execute.")


In [ ]:
# Fetch all and run wrangling inline (works even before TODOs are done)
records, offset = [], 0
while True:
    batch = requests.get(LAUNCHES_URL,
                         params={"offset": offset, "limit": 100}, timeout=15).json()
    if not batch:
        break
    records.extend(batch)
    offset += 100

df_check = pd.DataFrame([{
    "flight_number": r.get("flight_number"),
    "name":          r.get("name"),
    "date_utc":      r.get("date_utc"),
    "success":       r.get("success"),
} for r in records])
df_check["date_utc"] = pd.to_datetime(df_check["date_utc"], utc=True)

print(f"Records   : {len(df_check)}")
print(f"Date range: {df_check['date_utc'].min().date()} → {df_check['date_utc'].max().date()}")
print(f"Null success: {df_check['success'].isna().sum()}")
print()
# Uncomment after implementing coerce_boolean():
# df_check['success_bool'] = coerce_boolean(df_check['success'].astype(str))
# print('Success rate:', df_check['success_bool'].mean())
df_check.head()


---
## Summary

| Concept | Rule |
|---------|------|
| **Pagination** | Always write the loop. A full-looking response might be page one. |
| **Backoff** | `Retry(backoff_factor=1.0, respect_retry_after_header=True)` handles it. |
| **Schema validation** | Pydantic at ingest makes schema drift a loud error, not a silent null. |
| **Change detection** | Hash raw content. Different hash → raise before parsing. |
| **MNAR** | Missing-not-at-random → indicator columns, never impute. |
| **`errors='coerce'`** | Makes coercion failures visible (count NaNs) not silent. |
| **Raw vs processed** | `data/raw/` precious. `data/processed/` disposable. |
| **Idempotency** | Run twice = same output. Hash-gated fetches make this free. |

## Exercise connection

| What you built | Used downstream |
|---|---|
| `data/processed/launches_clean.parquet` | Input to `exercises/01-ml/01-regression/src/data-prep.py` |
| `validate_launches()` pattern | Template for `data-prep.py` TODOs #11–12 (PSI / KS tests) |
| `coerce_boolean()` | Extended in `data-prep.py` for SmartVal AI dataset |

**Reference:** [data-acquisition.md](data-acquisition.md)